# Verification of the MICADO Calibration Assembly (MCA)

This notebook describes how to simulate calibration exposures using the MICADO Calibration Assembly with ScopeSim. For this purpose a new submode `CALIB` has been introduced, which replaces the `SCAO` and `MCAO` submodes that are used for on-sky observations. It can be combined with all instrument modes, i.e. `IMG_4mas`, `SPEC`, etc. 
`CALIB` does not include the Armazones (atmosphere) and ELT effects. It currently includes the following to describe the MCA:
- `mca_mirror`: a single deployable mirror that is unique to the MCA. As with all mirrors the effect describes throughput (reflectivity) as well as thermal emission.
- `relay_surface_list`: This is the list of mirrors in the relay optics that is used in stand-alone mode, identical to the mirror list in the `SCAO` submode. Note that MORFEO is not yet supported for MCA simulations.
- `air_transmission`: The optical path from the MCA to the entrance window of MICADO has a length of about 14 metres through air, which therefore imprints an absorption signal on the input (continuum) spectrum. The effect uses a library of transmission spectra for various values of relative humidity (see below for details).
- `psf`: Very simplistically, the instrumental PSF (imprinted on observations using a pinhole mask) is modeled as Gaussian PSF of FWHM = 0.02 arcsec. This can be made more realistic in the future.

In [ ]:
import scopesim as sim

In [ ]:
sim.link_irdb("../../../")

If you have not done so already, please download the relevant instrument packages using the following code in a new cell:

```sim.download_packages(["MICADO"])```

Alternatively, if you would like to keep the instrument packages in a separate directory, you can set the following config value:

```sim.set_inst_pkgs_path("path/to/packages")```

In [ ]:
# sim.set_inst_pkgs_path("/Users/user/path/inst_pkgs")

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from astropy import units as u
from astropy.wcs import WCS

## Setting up the optical train
We first set up MICADO for the nominal imaging mode, and fix the "telescope area" to the MCA deployable mirror area:

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "IMG_4mas"]) #, properties={"!TEL.area": 0.20029616662043426})

Uncomment to include debug messages

In [ ]:
# from scopesim import set_console_log_level
# set_console_log_level("DEBUG")
# from scopesim.optics.image_plane_utils import logger
# logger.setLevel("DEBUG")

Instatiate the optical train and confirm the TEL.area

In [ ]:
micado = sim.OpticalTrain(cmd)
micado["filter_wheel_2"].change_filter("Ks")
# micado["filter_wheel_2"].change_filter("open")
micado.cmds["!TEL.area"]

### Check mirror areas
A snippet to print all the mirror areas in the effects list

In [ ]:
# micado.cmds["!TEL"]
# from scopesim.effects import SurfaceList
# list = micado.optics_manager.get_all(SurfaceList)
# for s in list:
#     print(s.display_name, s.area)

In [ ]:
micado.effects.pprint_all()

## Imaging 
The MICADO calibration mode needs to use a `Source` object (unlike the METIS WCU mode). We use a flat field from `Scopesim_Targets`.

First, we define the pixel scale of the *source* and its angular extent in *mas*

In [ ]:
from synphot import SourceSpectrum, units
from synphot.models import Empirical1D
from scopesim_targets.extended_source import Flat
from astropy.table import QTable, unique
from astropy.io import fits

PIXSCALE = 64.0 # pixel scale in mas
NX = int(1024 * 4 // PIXSCALE) # size of flat lamp in pixels
NY = int(1024 * 4 // PIXSCALE) # size of flat lamp in pixels

Read flux definition of the flat lamp from file...

In [ ]:
# COLUMN = "5V"
# ROOT = "../../../../flatcube/"

# tbl = unique(QTable.read(ROOT+"rad.ecsv"), "wavelength")
# wave = tbl["wavelength"]
# wave = wave.value.astype("float32") * wave.unit # convert to float32
# flux = tbl[COLUMN]
# flux

...or define it as a black-body spectrum

In [ ]:
from astropy.modeling.physical_models import BlackBody
from astropy import constants as c

bb = BlackBody(temperature = 2000*u.K)
wave = np.linspace(0.7, 2.5, 10000)*u.um
wave <<= u.nm
energy_flux_nu = bb(wave)

photon_energy = (c.h * c.c / wave).to(u.erg)

# Photon flux per Hz: (Energy Flux) / (Energy per Photon)
# Units: photon / (cm2 s Hz sr)
photon_flux_nu = (energy_flux_nu / photon_energy) * u.photon
photon_flux_lam = photon_flux_nu.to(
    u.photon / (u.cm**2 * u.s * u.AA * u.sr), 
    equivalencies=u.spectral_density(wave)
)
# arbitrary scaling factor
flux = photon_flux_lam * 1e-5 # * (1*u.arcsec**2)
flux <<= u.Unit("ph / (nm s sr cm2)")
flux 

### 2D source definition

In [ ]:
spec = SourceSpectrum(Empirical1D, points=wave, lookup_table=flux * (1*u.arcsec**2))

ref_wave = 1.75*u.um
ref_flux = spec(ref_wave).to(u.Jy, u.spectral_density(ref_wave))

tg = Flat(spectrum=spec, brightness=(ref_wave, ref_flux/u.arcsec**2))

grid = {
    "pixel_scale": (PIXSCALE * u.mas/u.pix) << u.arcsec/u.pix,
    "width": NX,
    "height": NY,
}

flat_2d = tg.to_source(grid)

In [ ]:
# flux at reference wavelength
spec(ref_wave)

### Cube source definition

Re-read the table from file, or comment out to use the previous flux definition

In [ ]:
# tbl = QTable.read(ROOT+"rad.ecsv")
# wave = tbl["wavelength"]
# flux = tbl[COLUMN]

In [ ]:
dwave = np.diff(wave)
delta_wave = dwave[0]
n_wave = len(wave)
pixel_scale = PIXSCALE * u.mas

cube = np.broadcast_to(
    flux.value.astype("float32")[:, None, None], (n_wave, NY, NX)
).copy()

w = WCS(naxis=3)
w.wcs.ctype = ["LINEAR", "LINEAR", "WAVE"]
w.wcs.cunit = ["deg", "deg", "nm"]
w.wcs.crpix = [NX / 2 + 0.5, NY / 2 + 0.5, 1]
w.wcs.crval = [0.0, 0.0, wave[0].value]
pix_scale_deg = pixel_scale.to(u.deg).value
w.wcs.cdelt = [pix_scale_deg, pix_scale_deg, delta_wave.value]

header = w.to_header()
header["BUNIT"] = flux.unit.to_string()
print(f"{header["BUNIT"] = }")

hdu = fits.PrimaryHDU(data=cube, header=header)
flat_cube = sim.Source(cube=hdu)

print(f"delta_wave: {delta_wave}")
print(f"pixel_area: {pixel_scale**2}")
print(f"field_area: {NX*NY*pixel_scale**2 << u.arcsec**2}")


## Back to the simulation

Plot the current filter's transmission

In [ ]:
micado["filter_wheel_2"].current_filter.plot();

In [ ]:
# Observe either the 2d or cube flat lamp source
micado.observe(flat_2d)

# Reference values for the BB spectrum

# 2D flat @ 16 mas pixels
# canvas_image_hdu.data.mean() = np.float64(55542064166.866264)
# After scaling: canvas_image_hdu.data.mean() = np.float64(888673.0266698446)

# 2D flat @ 32 mas pixels
# canvas_image_hdu.data.mean() = np.float64(55542064166.86521)
# After scaling: canvas_image_hdu.data.mean() = np.float64(888673.026669826)

# 3D flat @ 16 mas pixels
# canvas_image_hdu.data.mean() = np.float64(55541849853.236404)
# After scaling: canvas_image_hdu.data.mean() = np.float64(888669.5976517651)

# 3D flat @ 32 mas pixels
# canvas_image_hdu.data.mean() = np.float64(55541849853.236404)
# After scaling: canvas_image_hdu.data.mean() = np.float64(888669.5976517651)

### Flat souces comparison

First - inspect the properties of the 2D collapsed image

In [ ]:
from scopesim.source.source_fields import (
    HDUSourceField,
    ImageSourceField,
    CubeSourceField,
)

def flux_image(field: HDUSourceField):
    """Return 2D image in ph/s/cm2.

    Spectrum is integrated over whole range.
    Cubes are flattened and spectrally integrated.

    # usage e.g.:
    flximg = flux_image(micado._last_source.fields[0])
    """
    if isinstance(field, ImageSourceField):
        return field.data * field.spectrum.integrate()
    if isinstance(field, CubeSourceField):
        dlam = field.header["CDELT3"] * u.Unit(field.header["CUNIT3"])
        print(f"{field.bunit = }, {dlam = }, field.pixel_area = {field.pixel_area << u.mas**2 }")
        img = field.data.sum(axis=0) * field.bunit * dlam
        if field.is_bunit_spatially_differential:
            img *= field.pixel_area
        return img.to(u.ph/u.s/u.cm**2)
    raise TypeError(f"{type(field)} unsupported")

In [ ]:
# inspect either the source object or the source as ingested into the optical train

flximg = flux_image(flat_2d.fields[0])
print(f"flat_2d img shape {flximg.shape}")
print(f"flat_2d img mean {flximg.mean()}")

flximg = flux_image(flat_cube.fields[0])
print(f"flat_cube img shape {flximg.shape}")
print(f"flat_cube img mean {flximg.mean()}")

# flximg = flux_image(micado._last_source.fields[0])

# Referene values
# flat_2d: 142.18657012921608 ph / (s cm2) 
# flat_cube: 142.19518660314446 ph / (s cm2) 


In [ ]:
# check the header of the ingested source
# micado._last_source.fields[0].header

### Compare the spectra of the 2D and cube flat sources

In [ ]:
cube_field = flat_cube.fields[0]
flux = cube_field.data.mean(axis=(1,2)) * cube_field.bunit * cube_field.pixel_area << u.Unit("photlam")
print(f"{cube_field.bunit = }")
waveset = cube_field.waveset
spec = SourceSpectrum(Empirical1D, points=waveset, lookup_table=flux)
plt.plot(waveset, spec(waveset), label = "cube")

waveset_2d = flat_2d.fields[0].spectrum.waveset << u.um
plt.plot(waveset_2d, flat_2d.fields[0].spectrum(waveset_2d) / (NX*NY), label="2d")
plt.legend();

### Image plane check

In [ ]:
im = micado.image_planes[0].data
print("Sum:     ", im.sum() * u.Unit("ph/s"))
print("Max:     ", im.max() * u.Unit("ph/s/pix"))
print("Mean:     ", im.mean() * u.Unit("ph/s/pix"))
# plt.imshow(im)
# plt.colorbar();

# Reference values from BB source
# 2D_Flat @ 16 mas source pixel size


# 2D @ 32


# 3D @ 16


# 3D @ 32


### Readout check

In [ ]:
readout = micado.readout(dit=1, ndit=1)[0]

In [ ]:
print("Sum:     ", readout[1].data.sum())
print("Mean:     ", readout[1].data.mean())
print("Std. dev.:", readout[1].data.std())

In [ ]:
plt.hist(readout[1].data.ravel(), bins=100);

Also from scopesim_templates, a pinhole mask, which we shall set up as a regular grid for imaging (blindly taken from the documentation).

In [ ]:
from scopesim_templates.micado.pinhole_masks import pinhole_mask

In [ ]:
dr = np.arange(-5, 6, 0.5)      # [arcsec]
x, y = np.meshgrid(dr, dr)
x, y = x.flatten(), y.flatten()
waves = np.arange(0.7, 2.5, 0.001) * u.um
pinh = pinhole_mask(x, y, waves, sum_factor=9001)

In [ ]:
# micado.observe(pinh)

In [ ]:
# read_pinh = micado.readout(dit=1, ndit=1)[0]

In [ ]:
# plt.imshow(read_pinh[1].data)
# plt.title("Imaging, 4 mas");

In [ ]:
# cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "IMG_1.5mas"])

In [ ]:
# micado = sim.OpticalTrain(cmd)
# micado.observe(pinh)
# read_pinh_zoom = micado.readout(dit=1, ndit=1)[0]

In [ ]:
# plt.imshow(read_pinh_zoom[1].data)
# plt.title("Imaging, 1.5 mas");

# Spectroscopy

In [ ]:
cmd = sim.UserCommands(use_instrument="MICADO", set_modes=["CALIB", "SPEC"])
micado = sim.OpticalTrain(cmd)
cmd["!TEL.area"]

In [ ]:
micado.effects.pprint_all()

We will observe the flat lamp, which allows us to switch off the psf effects. We'll try to simulate a full field of view.

In [ ]:
micado['psf'].include = False
micado['micado_ncpas_psf'].include = False
micado['filter_wheel_1'].change_filter("Spec_HK")
micado['detector_window'].include = False
micado['full_detector_array'].include = True

The transmission of the 14 meter air column in the MCA and relay optics is provided by the `air_transmission` effect. This is a library of transmission spectra for relative humidities between 5 and 95 per cent, available in steps of 5 per cent. The default is 10 per cent, which can be changed with the `update()` method:

In [ ]:
air = micado['air_transmission']
print("Default humidity:", air.meta['relH'], "(per cent)")

In [ ]:
air.update(relH=21)
print("Current humidity:", air.meta['relH'], "(per cent)")

In [ ]:
air.plot();

In [ ]:
micado.observe(flat_cube)

# 2d @ 32, FOV1:
# 3D FOV make_imagefields: field_hdu.data.mean() = np.float64(0.059604644775390514) weightmap
# 3D FOV make_hdu: canvas_cube_hdu.data.mean() = np.float64(133029222573.6335)

# 2d @ 64, FOV1:
# 3D FOV make_imagefields: field_cube.mean() = <Quantity 1.36404382 PHOTLAM>
# 3D FOV make_hdu: canvas_cube_hdu.data.mean() = np.float64(133029222573.63048)

# 3d @ 32, FOV1:
# 3D FOV make_cubefields: field_data.mean() = np.float64(1.3640210615398614) ph s-1 arcsec-2
# 3D FOV make_hdu: canvas_cube_hdu.data.mean() = np.float64(133027002625.97089)

# 3d @ 64, FOV1:
# 3D FOV make_cubefields: field_data.mean() = np.float64(1.3640210615398596) PHOTLAM arcsec-2
# 3D FOV make_hdu: canvas_cube_hdu.data.mean() = np.float64(133027002625.97089)


In [ ]:
hdu = micado.fov_manager.fovs[0].view()

In [ ]:
hdu.writeto("spec_cube.fits")

In [ ]:
print(f"{micado.image_planes[0].data.max() = }")
# cube @32: 275.9863315908688 ph/s/pix
# cube @64: 275.9863315908688
# 2d @32: 275.98635377691744 ph/s/pix
# 2d @64: 275.98635377691744

# MCA 3d @64: 153.86593589531887

In [ ]:
micado.image_planes[0].hdu.writeto("spec_implane.fits", overwrite=True)

In [ ]:
readout = micado.readout(dit=60, ndit=1, filename="spec_readout.fits")[0]

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(readout[5].data)
# plt.imshow(micado.image_planes[0].data)
plt.colorbar();

In [ ]:
#rect = micado['micado_spectral_traces'].rectify_traces(readout, -1.5, 1.5)

In [ ]:
#plt.imshow(rect[1].data); 
#plt.xlim(500, 4000);

In [ ]:
# j = np.arange(rect[2].data.shape[1])
# wcs = WCS(rect[2].header).spectral
# lam = wcs.all_pix2world(j, 0)[0]
# lam = (lam * wcs.wcs.cunit[0]).to(u.um)
# plt.plot(lam, rect[2].data[300, ])
# plt.title(rect[2].header["EXTNAME"])
# plt.xlabel("Wavelength [um]");